# Calibration of logit PDs: Bayesian vs Platt

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

MODEL = '_imputed_logit'
SOURCE_SETS = ['train']
TARGET_SETS = ['oot1', 'oot2']

In [ ]:
scores = pd.concat([pd.read_csv(f'../Model_scores/scores_{s}{MODEL}.csv').assign(set=s)
                    for s in SOURCE_SETS + TARGET_SETS], ignore_index=True)
scores['year'] = pd.to_datetime(scores['stmt_date']).dt.year
source = scores['set'].isin(SOURCE_SETS)
target = scores['set'].isin(TARGET_SETS)

pi_s = scores.loc[source, 'y_true'].mean()
pi_t = scores.loc[target, 'y_true'].mean()
print(f'pi_s = {pi_s:.4%}  pi_t = {pi_t:.4%}')

pi_s = 1.3989%  pi_t = 1.4250%


In [ ]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def bayes(p, pi_s, pi_t):
    return 1 / (1 + np.exp(-(logit(p) + logit(pi_t) - logit(pi_s))))

scores = scores[target].copy()

platt = LogisticRegression(C=1e10, max_iter=1000)
platt.fit(logit(scores[['y_pred']].to_numpy()), scores['y_true'])
a, b = platt.intercept_[0], platt.coef_[0, 0]
print(f'Platt: a = {a:.4f}, b = {b:.4f}')

scores['pd_raw'] = scores['y_pred']
scores['pd_bayes'] = bayes(scores['y_pred'], pi_s, pi_t)
scores['pd_platt'] = 1 / (1 + np.exp(-(a + b * logit(scores['y_pred']))))
PDS = ['pd_raw', 'pd_bayes', 'pd_platt']

Platt: a = -0.0723, b = 0.9693


/Users/nikitaa/Documents/mlf_final/code_final/Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/nikitaa/Documents/mlf_final/code_final/Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/nikitaa/Documents/mlf_final/code_final/Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights


In [ ]:
def log_likelihood(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))

def check(g):
    y = g['y_true']
    out = {'firms': len(g), 'actual_defaults': y.sum()}
    for c in PDS:
        out[f'predicted_{c}'] = round(g[c].sum())
        out[f'LL_{c}'] = log_likelihood(y, g[c])
        out[f'AUC_{c}'] = roc_auc_score(y, g[c])
    for m in ['LL', 'AUC']:
        out[f'{m}_diff_bayes'] = out[f'{m}_pd_bayes'] - out[f'{m}_pd_raw']
        out[f'{m}_diff_platt'] = out[f'{m}_pd_platt'] - out[f'{m}_pd_raw']
    return pd.Series(out)

by_year = scores.groupby('year').apply(check, include_groups=False)
by_year.loc['total'] = check(scores)
for name, m in [('Log-likelihood (higher = better)', 'LL'), ('AUC', 'AUC')]:
    print(name); display(by_year[[f'{m}_{c}' for c in PDS] + [f'{m}_diff_bayes', f'{m}_diff_platt']].round(4))

Log-likelihood (higher = better)


,LL_pd_raw,LL_pd_bayes,LL_pd_platt,LL_diff_bayes,LL_diff_platt
year,,,,,
2010,-13418.4040,-13407.5998,-13391.8808,10.8042,26.5232
2011,-10274.0856,-10281.6982,-10292.2371,-7.6126,-18.1515
total,-23692.4896,-23689.2980,-23684.1178,3.1915,8.3717


AUC


,AUC_pd_raw,AUC_pd_bayes,AUC_pd_platt,AUC_diff_bayes,AUC_diff_platt
year,,,,,
2010,0.7606,0.7606,0.7606,0.0,0.0
2011,0.7820,0.7820,0.7820,0.0,0.0
total,0.7695,0.7695,0.7695,0.0,0.0


In [ ]:
scores.to_csv(f'../Model_scores/scores_target{MODEL}_calibrated.csv', index=False)